# Lab 08-01 · Number Systems and Codes

Companion to chapter [08-01](../../chapters/08-01-number-systems-and-codes.md). Handout: [README.md](README.md).

Every rule in the chapter is tested here on *all* the cases a small word can hold, not on a few examples: all 256 pairs of 4-bit numbers, every bit pattern of the 8-bit float formats, every error pattern of each code. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
from fractions import Fraction
import itertools
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(801)

## Part 1 · Bases

Integers convert by repeated division (the remainders are the digits, least significant first); fractions by repeated multiplication (the integer parts are the digits, most significant first). Exact `Fraction` arithmetic lets us watch a fraction repeat instead of guessing from rounded floats.

**🔮 Predict:** how many of 0.1, 0.2, …, 0.9 have a terminating binary expansion?

In [2]:
DIGITS = "0123456789ABCDEF"


def int_to_base(n, b):
    if n == 0:
        return "0"
    out = []
    while n:
        n, r = divmod(n, b)
        out.append(DIGITS[r])
    return "".join(reversed(out))


def frac_to_base(x, b, max_digits=24):
    # Digits of a fraction 0 <= x < 1 in base b, and whether the expansion terminated.
    x = Fraction(x)
    out = []
    for _ in range(max_digits):
        if x == 0:
            return "".join(out), True
        x *= b
        d = int(x)
        out.append(DIGITS[d])
        x -= d
    return "".join(out), x == 0


for n in (245, 1019, 2026):
    for b in (2, 8, 16):
        s = int_to_base(n, b)
        assert int(s, b) == n          # Python's own parser agrees
    print(f"{n:5d} = {int_to_base(n, 2):>12s}_2 = {int_to_base(n, 8):>5s}_8 = {int_to_base(n, 16):>4s}_16")

print()
terminating = []
for k in range(1, 10):
    digits, done = frac_to_base(Fraction(k, 10), 2)
    print(f"0.{k} -> 0.{digits}{'' if done else '...'}")
    if done:
        terminating.append(k)
print("terminating:", [f"0.{k}" for k in terminating])
assert terminating == [5]

  245 =     11110101_2 =   365_8 =   F5_16
 1019 =   1111111011_2 =  1773_8 =  3FB_16
 2026 =  11111101010_2 =  3752_8 =  7EA_16

0.1 -> 0.000110011001100110011001...
0.2 -> 0.001100110011001100110011...
0.3 -> 0.010011001100110011001100...
0.4 -> 0.011001100110011001100110...
0.5 -> 0.1
0.6 -> 0.100110011001100110011001...
0.7 -> 0.101100110011001100110011...
0.8 -> 0.110011001100110011001100...
0.9 -> 0.111001100110011001100110...
terminating: ['0.5']


A fraction terminates in base 2 only if its denominator, in lowest terms, is a power of 2. Every $k/10$ except $5/10 = 1/2$ keeps a factor 5 in the denominator, so its binary digits repeat forever (0.1 = 0.000110011…₂). That is why `0.1 + 0.2 != 0.3` in every language that uses binary floating point.

## Part 2 · Two's complement and overflow

A 4-bit register holds 16 patterns. Two's complement reads patterns 1000 to 1111 as −8 to −1, so the patterns sit on a wheel: adding moves clockwise, and overflow is crossing the seam between +7 and −8.

**🔮 Predict:** of the 256 ordered pairs $(a, b)$ of 4-bit two's-complement numbers, how many sums overflow?

In [3]:
N = 4


def signed(u, n=N):
    return u - (1 << n) if u >> (n - 1) else u


fig, ax = plt.subplots(figsize=(5, 5))
for u in range(16):
    ang = math.pi / 2 - 2 * math.pi * u / 16
    x, y = math.cos(ang), math.sin(ang)
    ax.plot(x, y, "o", color="#0072B2" if signed(u) >= 0 else "#D55E00", ms=10)
    ax.text(1.22 * x, 1.22 * y, f"{u:04b}\n{signed(u):+d}", ha="center", va="center", fontsize=8)
seam = math.pi / 2 - 2 * math.pi * 7.5 / 16
ax.plot([0, 1.1 * math.cos(seam)], [0, 1.1 * math.sin(seam)], "k--")
ax.text(0.35 * math.cos(seam) - 0.05, 0.35 * math.sin(seam) + 0.15, "overflow\nseam", fontsize=8)
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("4-bit two's complement: blue non-negative, orange negative")
plt.show()

In [4]:
def add_with_carries(a, b, n=N):
    # Ripple-carry addition of two n-bit patterns; returns (sum pattern, c_{n-1}, c_n).
    c, carries, s = 0, [0], 0
    for i in range(n):
        ai, bi = (a >> i) & 1, (b >> i) & 1
        s |= (ai ^ bi ^ c) << i
        c = (ai & bi) | (c & (ai ^ bi))
        carries.append(c)
    return s, carries[n - 1], carries[n]


overflows = 0
for ua, ub in itertools.product(range(16), repeat=2):
    a, b = signed(ua), signed(ub)
    s, c_in_top, c_out = add_with_carries(ua, ub)
    true_overflow = not (-8 <= a + b <= 7)
    sign_rule = (a < 0) == (b < 0) and (signed(s) < 0) != (a < 0)
    carry_rule = c_in_top != c_out
    assert sign_rule == true_overflow and carry_rule == true_overflow
    if not true_overflow:
        assert signed(s) == a + b          # without overflow the 4-bit result is exact
    overflows += true_overflow
print("overflowing pairs:", overflows, "of 256")
assert overflows == 64

overflowing pairs: 64 of 256


Both rules agree with the true sum on all 256 pairs, and the non-overflowing results are exact: two's-complement addition *is* ordinary addition, modulo 16. The 64 overflows split 28 too positive and 36 too negative, because the range −8 … +7 is lopsided.

## Part 3 · The 8-bit float formats

Decode every bit pattern of E4M3 and E5M2 from the field definitions (bias $2^{e-1}-1$; exponent field 0 means subnormal; E5M2 reserves the all-ones exponent for infinity and NaN like IEEE; E4M3 reserves only S.1111.111 for NaN).

**🔮 Predict:** how many of E4M3's 256 patterns are finite numbers, and where is its spacing exactly 1?

In [5]:
def decode(pattern, e, m, kind):
    sgn = -1 if pattern >> (e + m) else 1
    E = (pattern >> m) & ((1 << e) - 1)
    F = pattern & ((1 << m) - 1)
    bias = (1 << (e - 1)) - 1
    if kind == "ieee" and E == (1 << e) - 1:
        return None                                      # inf or NaN
    if kind == "e4m3" and E == (1 << e) - 1 and F == (1 << m) - 1:
        return None                                      # the single NaN pattern
    if E == 0:
        return sgn * Fraction(F, 1 << m) * Fraction(2) ** (1 - bias)
    return sgn * (1 + Fraction(F, 1 << m)) * Fraction(2) ** (E - bias)


fmts = {"E4M3": (4, 3, "e4m3"), "E5M2": (5, 2, "ieee")}
values = {}
for name, (e, m, kind) in fmts.items():
    vals = [decode(p, e, m, kind) for p in range(256)]
    finite = [v for v in vals if v is not None]
    values[name] = sorted({v for v in finite if v > 0})
    print(f"{name}: {len(finite)} finite patterns, {len(values[name])} positive values, "
          f"largest {float(max(finite))}, smallest positive {float(values[name][0]):.6g}")
assert len([p for p in range(256) if decode(p, 4, 3, "e4m3") is not None]) == 254
assert max(values["E4M3"]) == 448 and max(values["E5M2"]) == 57344

# where is E4M3's spacing exactly 1?
v = values["E4M3"]
ones = [(float(a), float(b)) for a, b in zip(v, v[1:]) if b - a == 1]
print("E4M3 spacing 1 between", ones[0][0], "and", ones[-1][1])
assert ones[0][0] == 8 and ones[-1][1] == 16

E4M3: 254 finite patterns, 126 positive values, largest 448.0, smallest positive 0.00195312
E5M2: 248 finite patterns, 123 positive values, largest 57344.0, smallest positive 1.52588e-05
E4M3 spacing 1 between 8.0 and 16.0


In [6]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))
for name, color in (("E4M3", "#D55E00"), ("E5M2", "#009E73")):
    v = np.array([float(x) for x in values[name]])
    ax.step(v[:-1], np.diff(v), where="post", color=color, label=name)
x16 = np.logspace(-4, 4.8, 400).astype(np.float16)
ax.step(x16.astype(float), np.spacing(x16).astype(float), where="post", color="#0072B2", label="FP16")
xb = np.logspace(-4, 4.8, 400)
ax.step(xb, 2.0 ** (np.floor(np.log2(xb)) - 7), where="post", color="#CC79A7", label="bfloat16")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("magnitude"); ax.set_ylabel("gap to the next value")
ax.set_title("Spacing of representable values"); ax.legend()
plt.show()

On log–log axes each normal range is a staircase of slope 1: the gap doubles every octave, so the *relative* spacing is constant ($2^{-m}$). The subnormals are the flat part at the left, where the gap stops shrinking. E4M3 buys finer steps than E5M2 (3 fraction bits against 2) and pays in range: it stops at 448, E5M2 at 57 344. bfloat16 trades the other way: 7 fraction bits, coarser than FP16's 10, but FP32's range, which runs far beyond both ends of this plot.

## Part 4 · A noisy channel (the chapter's guided exercise)

Each bit of a code word flips independently with probability $p$. A **silent failure** is a word the receiver accepts with wrong data and no flag. The four schemes (see the handout): parity on 8 data bits (9 bits), Hamming(7,4) SEC, SEC-DED (8 bits), and the CRC $x^3+x+1$ on 4-bit messages used for detection only.

**🔮 Predict:** when $p$ rises from $10^{-3}$ to $10^{-2}$, by what factor does each scheme's silent-failure probability rise?

Because the codes are linear, whether a received word fails depends only on the error pattern. So we can push *every* error pattern through each decoder and add up $p^w(1-p)^{n-w}$ over the failing ones: an exact answer, no sampling.

In [7]:
H = np.array([[(k >> r) & 1 for k in range(1, 8)] for r in range(3)])   # column k is k in binary
DATA_POS = [3, 5, 6, 7]


def ham_encode(d):
    c = np.zeros(7, dtype=int)
    for pos, bit in zip(DATA_POS, d):
        c[pos - 1] = bit
    for r, ppos in enumerate((1, 2, 4)):
        c[ppos - 1] = (H[r] @ c) % 2
    return c


def sec_decode(r):
    syn = (H @ r) % 2
    pos = int(syn[0] + 2 * syn[1] + 4 * syn[2])
    rr = r.copy()
    if pos:
        rr[pos - 1] ^= 1
    return [int(rr[k - 1]) for k in DATA_POS], pos


def clmod(a, g):
    # remainder of carry-less (GF(2)) division
    while a.bit_length() >= g.bit_length():
        a ^= g << (a.bit_length() - g.bit_length())
    return a


def bits(e, n):
    return np.array([(e >> (n - 1 - k)) & 1 for k in range(n)])


data = [1, 0, 1, 1]
cw7 = ham_encode(data)
cw8 = np.append(cw7, cw7.sum() % 2)
crc_word = (0b1101 << 3) | clmod(0b1101 << 3, 0b1011)


def silent_parity(e):
    return e != 0 and bin(e).count("1") % 2 == 0


def silent_sec(e):
    return sec_decode((cw7 + bits(e, 7)) % 2)[0] != data


def secded(e):
    r = (cw8 + bits(e, 8)) % 2
    d, pos = sec_decode(r[:7])
    flagged = bool(pos) and r.sum() % 2 == 0
    return flagged, (not flagged) and d != data


def silent_crc(e):
    return e != 0 and clmod(crc_word ^ e, 0b1011) == 0


schemes = {"parity (9 bits)": (9, silent_parity), "SEC (7,4)": (7, silent_sec),
           "SEC-DED (8,4)": (8, lambda e: secded(e)[1]), "CRC x^3+x+1 (7)": (7, silent_crc)}


def exact(n, silent, p):
    return sum(p ** bin(e).count("1") * (1 - p) ** (n - bin(e).count("1"))
               for e in range(2 ** n) if silent(e))


print(f"{'scheme':18s} {'p = 1e-3':>11s} {'p = 1e-2':>11s} {'ratio':>7s}  first failing weight")
results = {}
for name, (n, silent) in schemes.items():
    lo, hi = exact(n, silent, 1e-3), exact(n, silent, 1e-2)
    wmin = min(bin(e).count("1") for e in range(2 ** n) if silent(e))
    results[name] = (lo, hi)
    print(f"{name:18s} {lo:11.4g} {hi:11.4g} {hi / lo:7.1f}  {wmin}")
flag = exact(8, lambda e: secded(e)[0], 1e-3)
print(f"\nSEC-DED words flagged (not silent) at p = 1e-3: {flag:.4g}")
assert abs(results["SEC-DED (8,4)"][0] / 5.573e-8 - 1) < 1e-3
assert abs(results["CRC x^3+x+1 (7)"][1] / 6.792e-6 - 1) < 1e-3

scheme                p = 1e-3    p = 1e-2   ratio  first failing weight
parity (9 bits)      3.575e-05    0.003357    93.9  2
SEC (7,4)            2.093e-05    0.002031    97.0  2
SEC-DED (8,4)        5.573e-08    5.34e-05   958.0  3
CRC x^3+x+1 (7)      6.979e-09   6.792e-06   973.2  3

SEC-DED words flagged (not silent) at p = 1e-3: 2.783e-05


The factors land near $10^{w}$, where $w$ is the smallest number of bit errors that can cause a silent failure: about 100 for parity and SEC ($w = 2$), about 1000 for SEC-DED and the CRC ($w = 3$). Notice the SEC and CRC rows use the *same 16 code words*: detection alone is about 3000 times safer at $p = 10^{-3}$ than correction, because a corrector confidently "repairs" every double error into a wrong word.

**Monte Carlo check.** The exact sums can be confirmed by simulation, but only where failures are frequent enough to count. At $p = 10^{-3}$ SEC-DED fails about once in 18 million words, so we test at larger $p$, where the same formulas apply.

In [8]:
def monte_carlo(n, silent, p, words=200_000):
    errs = rng.random((words, n)) < p
    patterns = errs @ (1 << np.arange(n - 1, -1, -1))
    table = np.array([silent(e) for e in range(2 ** n)])   # silent-failure lookup for every pattern
    return table[patterns].mean()


for p in (0.02, 0.05):
    print(f"p = {p}")
    for name, (n, silent) in schemes.items():
        ex = exact(n, silent, p)
        mc = monte_carlo(n, silent, p)
        sigma = math.sqrt(ex * (1 - ex) / 200_000)
        print(f"  {name:18s} exact {ex:.4g}   simulated {mc:.4g}   ({(mc - ex) / sigma:+.1f} sigma)")
        assert abs(mc - ex) < 5 * sigma + 1e-12

p = 0.02
  parity (9 bits)    exact 0.01252   simulated 0.01248   (-0.2 sigma)
  SEC (7,4)          exact 0.007857   simulated 0.00769   (-0.8 sigma)
  SEC-DED (8,4)      exact 0.0004072   simulated 0.00042   (+0.3 sigma)
  CRC x^3+x+1 (7)    exact 5.271e-05   simulated 8e-05   (+1.7 sigma)
p = 0.05
  parity (9 bits)    exact 0.06346   simulated 0.06374   (+0.5 sigma)


  SEC (7,4)          exact 0.04438   simulated 0.04448   (+0.2 sigma)
  SEC-DED (8,4)      exact 0.005503   simulated 0.00583   (+2.0 sigma)
  CRC x^3+x+1 (7)    exact 0.0007502   simulated 0.000745   (-0.1 sigma)


The simulation agrees with the exact sums to within the expected statistical scatter. That is the general method for qualifying very reliable systems: measure at stressed conditions where failures are countable, check the model there, and use the model (not brute force) at the operating point.

## What to take away

1. Base conversion is division and multiplication; most decimal fractions are infinite in binary.
2. Two's complement is arithmetic modulo $2^n$; overflow is crossing the seam, detected by either the sign rule or the carry rule.
3. A float format's spacing grows with magnitude; the fraction width sets the relative step, the exponent width the range.
4. A code's silent-failure rate scales as $p^{w}$, with $w$ set by its minimum distance and by whether it corrects or only detects.